# Ingeniería de variables: construir y verificar variables nuevas

## Qué responde este notebook

Los notebooks 2.1 a 2.6 describieron qué variables existen y cómo se comportan. Este notebook
**construye** variables nuevas a partir de ahí: razones de eficiencia, un tope para `racr`,
volatilidad reciente, transformaciones de la experiencia, el número de draft, un indicador de cambio
de quarterback titular, la ofensiva del equipo y combinaciones con una razón concreta para probarse.

La pregunta es si cada función calcula lo que dice calcular, sin fuga de datos; cada una se verifica
fila por fila sobre un caso real, con el criterio de
[`1.4_promedios_sin_fuga.ipynb`](1.4_promedios_sin_fuga.ipynb). Cuáles de estas variables ayudan a
predecir se evalúa en [`3.2_seleccion_features.ipynb`](3.2_seleccion_features.ipynb). Las variables
se definen en el [glosario](../../../docs/data/GLOSARIO_VARIABLES.md), secciones 7 y 9.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, features_exploratorio

seasons = list(range(2016, 2026))
stats = data.cargar_stats_semanales(seasons)
jugadores = data.cargar_jugadores()
wr = stats[stats["position"] == "WR"].copy()
print("temporadas:", seasons[0], "-", seasons[-1])
print("filas WR:", wr.shape[0])

temporadas: 2016 - 2025
filas WR: 23610


## 1. Razones de eficiencia: un ejemplo antes que la fórmula

La tabla de estadísticas trae volúmenes (yardas, recepciones). Falta la eficiencia: de lo que le
llegó, cuánto convirtió.

Ja'Marr Chase, semana 1 de 2025: le dirigieron **5 pases**, atrapó **2** y sumó **26 yardas**; en esos
5 pases el balón viajó **88 yardas** en el aire en total. De ahí salen tres preguntas:

- **¿Qué tan seguido atrapa lo que le lanzan?** `catch_rate` = 2 / 5 = **0.40**.
- **¿Cuántas yardas produce por pase dirigido?** `yards_per_target` = 26 / 5 = **5.2**.
- **¿Qué tan profundos son los pases que le lanzan?** `air_yards_per_target` = 88 / 5 = **17.6**.

`features.calcular_ratios_eficiencia` calcula las tres.

In [2]:
wr = features.calcular_ratios_eficiencia(wr)
chase = wr[(wr["player_display_name"] == "Ja'Marr Chase") & (wr["season"] == 2025) & (wr["week"] == 1)]
chase[["player_display_name", "targets", "receptions", "receiving_yards", "receiving_air_yards",
       "catch_rate", "yards_per_target", "air_yards_per_target"]]

,player_display_name,targets,receptions,receiving_yards,receiving_air_yards,catch_rate,yards_per_target,air_yards_per_target
156313,Ja'Marr Chase,5,2,26,88,0.4,5.2,17.6


Las tres coinciden con la cuenta de arriba. Falta un caso al margen: una semana sin pases dirigidos.
Dividir entre 0 no debe dar error ni inventar un 0: un receptor sin pases dirigidos no tiene tasa de
acierto, y el valor correcto es «no aplica».

In [3]:
sin_targets = wr[wr["targets"] == 0]
print("filas con targets=0:", sin_targets.shape[0])
print("catch_rate nulo en todas:", sin_targets["catch_rate"].isna().all())
print("yards_per_target nulo en todas:", sin_targets["yards_per_target"].isna().all())

filas con targets=0: 3110
catch_rate nulo en todas: True
yards_per_target nulo en todas: True


En las 3,110 filas sin pases dirigidos, `catch_rate` y `yards_per_target` quedan vacías, nunca en 0
ni con error. Así, al promediarlas después, el promedio de un jugador no se distorsiona con ceros
falsos.

## 2. `racr` con tope

[`2.2_eda_perfil_variables.ipynb`](2.2_eda_perfil_variables.ipynb) mostró que `racr` (yardas de
recepción entre yardas aéreas) es numéricamente inestable: cuando el denominador se acerca a cero,
el cociente se dispara. El caso más extremo fue Khalil Shakir, semana 9 de 2025: 43 yardas de
recepción sobre 1 yarda aérea, `racr` = 43.0.

En lugar de descartar `racr`, se prueba una versión con tope en el percentil 99 de la columna
(calculado sobre los datos, no un número fijo). Si aporta algo se decide en
[`3.2_seleccion_features.ipynb`](3.2_seleccion_features.ipynb).

In [4]:
wr["racr"] = (wr["receiving_yards"] / wr["receiving_air_yards"]).where(wr["receiving_air_yards"] > 0)
shakir = wr[(wr["player_display_name"] == "Khalil Shakir") & (wr["season"] == 2025) & (wr["week"] == 9)]
print(shakir[["player_display_name", "receiving_yards", "receiving_air_yards", "racr"]].to_string(index=False))

wr = features_exploratorio.acotar_variable(wr, "racr", percentil=0.99)
tope = wr["racr"].quantile(0.99)
print(f"\npercentil 99 real de racr: {tope:.3f}")
shakir2 = wr[(wr["player_display_name"] == "Khalil Shakir") & (wr["season"] == 2025) & (wr["week"] == 9)]
print(shakir2[["racr", "racr_acotado"]].to_string(index=False))

player_display_name  receiving_yards  receiving_air_yards  racr
      Khalil Shakir               43                    1  43.0

percentil 99 real de racr: 5.667
 racr  racr_acotado
 43.0      5.666667


El valor de 43.0 queda topado en 5.667, el percentil 99: deja de ser un extremo capaz de arrastrar el
promedio de un jugador, sin descartar la fila.

## 3. Volatilidad reciente: además del nivel, la consistencia

Dos receptores pueden tener el mismo promedio y comportarse muy distinto: uno produce lo mismo cada
semana y otro alterna semanas enormes con semanas casi en cero, como la de Chase en
[`2.6_eda_jugador_destacado.ipynb`](2.6_eda_jugador_destacado.ipynb), casi al triple de su promedio.
Un promedio no distingue esos dos casos.

Se agrega la desviación estándar de las últimas 5 apariciones, rezagada igual que
`agregar_promedios_jugador`: un valor más alto significa más irregularidad.

In [5]:
wr = features_exploratorio.agregar_volatilidad_jugador(wr, ["receiving_yards"], ventana=5)
chase = wr[(wr["player_display_name"] == "Ja'Marr Chase") & (wr["season"] == 2025)].sort_values("week")
chase[["week", "receiving_yards", "receiving_yards_volatilidad5"]].head(8)

,week,receiving_yards,receiving_yards_volatilidad5
156313,1,26,35.786869
157388,2,165,32.000000
158485,3,50,49.220931
159593,4,23,53.564914
160637,5,110,59.636398
161620,6,94,61.349002
162644,7,161,55.039077
163614,8,91,53.659109


La primera fila de 2025 ya trae un valor, porque la ventana de 5 apariciones no se reinicia por
temporada, igual que `_last5_avg`: arrastra las últimas semanas de 2024.

## 4. Edad y experiencia: la transformación no lineal

[`2.3_eda_general.ipynb`](2.3_eda_general.ipynb) mostró que la correlación lineal de la experiencia con
los tres resultados es prácticamente cero, pero la relación tiene forma de curva: sube hasta 3-5 años
y baja después. Un término cuadrático o una categoría por grupos de experiencia le dan a un modelo
lineal la oportunidad de captar esa curva.

Se agregan `edad` y `anios_experiencia` (a partir de la tabla de jugadores), el cuadrado de la
experiencia y la categoría por grupos, con los mismos cortes de 2.3 (`Rookie`, `1-2 años`, `3-5 años`,
`6+ años`).

In [6]:
wr = features.agregar_edad_experiencia(wr, jugadores)
wr = features.agregar_experiencia_no_lineal(wr)

carrera_larga = wr[wr["anios_experiencia"] >= 10].drop_duplicates("player_display_name")
carrera_larga = carrera_larga.sort_values("anios_experiencia", ascending=False)
carrera_larga[["player_display_name", "anios_experiencia", "anios_experiencia_sq",
               "anios_experiencia_bucket", "edad"]].head(5)

,player_display_name,anios_experiencia,anios_experiencia_sq,anios_experiencia_bucket,edad
0,Steve Smith,15,225,6+ años,37.308693
20,Anquan Boldin,13,169,6+ años,35.912389
14,Andre Johnson,13,169,6+ años,35.143053
36,Larry Fitzgerald,12,144,6+ años,33.004791
113,Vincent Jackson,11,121,6+ años,33.631759


Jugadores de carrera larga, como Steve Smith y Larry Fitzgerald, caen en `6+ años`, con el cuadrado
de su experiencia calculado sobre su valor.

## 5. `draft_pick` y los jugadores no seleccionados

[`2.5_eda_multivariable.ipynb`](2.5_eda_multivariable.ipynb) encontró que `draft_pick` se relaciona
más con la producción que la edad o la experiencia (−0.22 con recepciones, −0.21 con yardas). El
problema es que viene vacío para los receptores que no fueron seleccionados en el draft (40.3% de los
receptores de la tabla de modelado, según el glosario, sección 7.3), y ese vacío no es un dato
faltante: es la ausencia real de una selección.

Imputar con la media o la mediana trataría a un no seleccionado como a uno «promedio», y no lo es: no
ser seleccionado es peor que la última selección. Se imputa como **la peor selección real + 1**,
calculada sobre los datos.

In [7]:
wr = features.agregar_draft_pick(wr, jugadores)
peor_que_ultimo = jugadores["draft_pick"].max() + 1
print(f"peor pick real + 1 = {peor_que_ultimo}")

chase = wr[(wr["player_display_name"] == "Ja'Marr Chase")].iloc[0]
print(f"\nJa'Marr Chase (sí drafteado, pick real conocido: 5): draft_pick = {chase['draft_pick']}")

no_drafteados = wr[wr["draft_pick"] == peor_que_ultimo]
ejemplo = no_drafteados.iloc[0]
print(f"{ejemplo['player_display_name']} (no drafteado, {no_drafteados['player_id'].nunique()} "
      f"jugadores distintos en este caso): draft_pick imputado = {ejemplo['draft_pick']}")

peor pick real + 1 = 473.0

Ja'Marr Chase (sí drafteado, pick real conocido: 5): draft_pick = 5.0
Eric Weems (no drafteado, 282 jugadores distintos en este caso): draft_pick imputado = 473.0


Chase conserva su selección real (5) y a los 282 jugadores no seleccionados se les asigna 473: peor
que cualquier selección real de los datos.

## 6. Cambio de quarterback titular: revisar los casos límite antes de construir

En [`2.5_eda_multivariable.ipynb`](2.5_eda_multivariable.ipynb), un cambio de quarterback (QB)
titular coincidió con unos 16% menos de yardas, más de lo que se relaciona la calidad continua del
QB. Ahí se usó un prototipo: el pasador con más intentos de pase esa semana en cada equipo. Antes de
convertirlo en una función se revisan tres casos límite con datos.

**a) ¿Qué tan seguido empatan dos pasadores del mismo equipo en la misma semana?**

In [8]:
qb = stats[(stats["position"] == "QB") & (stats["attempts"] > 0)][
    ["season", "week", "team", "player_display_name", "attempts", "passing_epa"]
].copy()
grp = qb.groupby(["season", "week", "team"])
maximo = grp["attempts"].transform("max")
empatados_en_max = grp["attempts"].transform(lambda s: (s == s.max()).sum())
empates = qb[(qb["attempts"] == maximo) & (empatados_en_max > 1)]
print(f"equipo-semana con empate en el máximo de attempts: {empates.shape[0] // 2} de {grp.ngroups} "
      f"({100 * (empates.shape[0] // 2) / grp.ngroups:.2f}%)")
empates.sort_values(["season", "week", "team"]).head(6)

equipo-semana con empate en el máximo de attempts: 7 de 5269 (0.13%)


,season,week,team,player_display_name,attempts,passing_epa
19952,2017,4,TEN,Matt Cassel,10,-22.404021
20666,2017,4,TEN,Marcus Mariota,10,-1.115625
105380,2022,4,PIT,Mitchell Trubisky,13,-6.632850
106078,2022,4,PIT,Kenny Pickett,13,1.425465
126824,2023,8,NYG,Tyrod Taylor,7,-10.349501
127736,2023,8,NYG,Tommy DeVito,7,-6.947000


Es rarísimo (7 de 5,269 equipo-semana, 0.13%), pero pasa. Se desempata por `passing_epa` (quién jugó
mejor esa semana), no por el orden en que vienen las filas.

**b) ¿Hay cambios de titular dentro de una misma temporada?**

In [9]:
qb_sorted = qb.sort_values(["attempts", "passing_epa"], ascending=False)
qb_starter = qb_sorted.drop_duplicates(subset=["season", "week", "team"]).sort_values(["team", "season", "week"])
qb_starter["titular_anterior"] = qb_starter.groupby("team")["player_display_name"].shift(1)
qb_starter["semana_anterior"] = qb_starter.groupby("team")["week"].shift(1)
qb_starter["temporada_anterior"] = qb_starter.groupby("team")["season"].shift(1)
cambios = qb_starter[
    (qb_starter["player_display_name"] != qb_starter["titular_anterior"])
    & qb_starter["titular_anterior"].notna()
    & (qb_starter["season"] == qb_starter["temporada_anterior"])
]
print(f"cambios de titular reales (misma temporada): {cambios.shape[0]}")

cambios de titular reales (misma temporada): 532


532 cambios en 10 temporadas: no es un evento anecdótico, vale la pena construir la variable.

**c) ¿Un traspaso del receptor a otro equipo se confunde con un cambio de QB?**

In [10]:
wr_sorted = wr.sort_values(["player_id", "season", "week"])
wr_sorted["team_anterior"] = wr_sorted.groupby("player_id")["team"].shift(1)
wr_sorted["season_anterior"] = wr_sorted.groupby("player_id")["season"].shift(1)
traspasos = wr_sorted[
    (wr_sorted["team"] != wr_sorted["team_anterior"])
    & wr_sorted["team_anterior"].notna()
    & (wr_sorted["season"] == wr_sorted["season_anterior"])
]
print(f"filas de WR con cambio de equipo dentro de la misma temporada: {traspasos.shape[0]}")

filas de WR con cambio de equipo dentro de la misma temporada: 105


Hay 105 filas de WR con cambio de equipo dentro de la misma temporada. Si se compara contra la
aparición anterior del receptor, un traspaso también se leería como «cambió el QB titular», porque
casi siempre lo es. Por eso se construyen dos indicadores separados, `cambio_qb_titular` y
`cambio_equipo`, para no mezclar dos señales en una sola variable.

### Construcción, verificada sobre dos casos

**Cleveland, 2024**: en la semana 7 el titular pasó de Deshaun Watson a Dorian Thompson-Robinson, y
en la 8 a Jameis Winston. El cambio debe marcarse en las semanas 7 y 8 para cada receptor del equipo,
y en ninguna otra (la tabla muestra las semanas 6 a 9).

In [11]:
qb_titular = data.identificar_qb_titular(stats)
wr = features_exploratorio.agregar_cambio_qb_titular(wr, qb_titular)

cle = wr[(wr["season"] == 2024) & (wr["team"] == "CLE") & (wr["week"].between(6, 9))]
cle.sort_values(["player_display_name", "week"])[
    ["player_display_name", "week", "qb_nombre", "cambio_qb_titular", "cambio_equipo"]
]

,player_display_name,week,qb_nombre,cambio_qb_titular,cambio_equipo
5871,Amari Cooper,6,Deshaun Watson,False,False
22126,Cedric Tillman,6,Deshaun Watson,False,False
22127,Cedric Tillman,7,Dorian Thompson-Robinson,True,False
22128,Cedric Tillman,8,Jameis Winston,True,False
22129,Cedric Tillman,9,Jameis Winston,False,False
19660,Elijah Moore,6,Deshaun Watson,False,False
19661,Elijah Moore,7,Dorian Thompson-Robinson,True,False
19662,Elijah Moore,8,Jameis Winston,True,False
19663,Elijah Moore,9,Jameis Winston,False,False
19224,Jaelon Darden,6,Deshaun Watson,False,False


Es lo esperado para los cuatro receptores que jugaron esas semanas: `cambio_qb_titular` = True en la
semana 7 (Watson → Thompson-Robinson) y en la 8 (→ Winston), False en las demás, y `cambio_equipo` =
False en todas, porque ninguno cambió de equipo.

**Ke'Shawn Williams, 2025**: pasó de Pittsburgh (semana 13) a Cincinnati (semana 16). Ahí debe
encenderse `cambio_equipo` y, casi seguro, también `cambio_qb_titular`.

In [12]:
williams = wr[wr["player_display_name"] == "Ke'Shawn Williams"].sort_values(["season", "week"])
williams[williams["season"] == 2025][["week", "team", "qb_nombre", "cambio_qb_titular", "cambio_equipo"]]

,week,team,qb_nombre,cambio_qb_titular,cambio_equipo
23441,6,PIT,Aaron Rodgers,NaN,NaN
23442,7,PIT,Aaron Rodgers,False,False
23443,8,PIT,Aaron Rodgers,False,False
23444,9,PIT,Aaron Rodgers,False,False
23445,10,PIT,Aaron Rodgers,False,False
23446,11,PIT,Mason Rudolph,True,False
23447,12,PIT,Mason Rudolph,False,False
23448,13,PIT,Aaron Rodgers,True,False
23449,16,CIN,Joe Burrow,True,True
23450,17,CIN,Joe Burrow,False,False


En la semana 16, `cambio_equipo` y `cambio_qb_titular` son True a la vez; antes, en las semanas 11 y
13, solo se enciende `cambio_qb_titular` (Rodgers → Rudolph → Rodgers). Los dos indicadores separan lo
que de verdad pasó.

## 7. Ofensiva del equipo: verificación de no fuga con aritmética

[`2.4_eda_equipos.ipynb`](2.4_eda_equipos.ipynb) encontró que la ofensiva de un equipo es
relativamente estable de una temporada a otra (0.40), y
[`2.5_eda_multivariable.ipynb`](2.5_eda_multivariable.ipynb), que su versión dentro de la temporada
se relaciona poco con la producción de cada receptor (0.08). Aquí se construye esa versión semanal:
las yardas de recepción de todos los WR del equipo, promediadas con las mismas ventanas y sin el
partido actual, reutilizando la lógica de `agregar_promedios_jugador()`.

La prueba de no fuga es aritmética: sumar a mano las yardas del equipo en sus primeras semanas,
dividir entre el número de semanas y comparar con lo que calcula la función.

In [13]:
wr = features_exploratorio.agregar_ofensiva_equipo(wr, columnas=("receiving_yards",), ventanas=(3, 5))

cin_1a4 = wr[(wr["team"] == "CIN") & (wr["season"] == 2025) & (wr["week"] < 5)]
suma_manual = cin_1a4.drop_duplicates(["week"])  # evitar doble conteo si un jugador aparece 2 veces
suma_manual = cin_1a4.groupby("week")["receiving_yards"].sum()
promedio_manual = suma_manual.sum() / suma_manual.shape[0]
print(f"suma manual de receiving_yards de CIN, semanas 1-4: {suma_manual.sum()}, "
      f"entre {suma_manual.shape[0]} semanas = {promedio_manual:.2f}")

semana_5 = wr[(wr["team"] == "CIN") & (wr["season"] == 2025) & (wr["week"] == 5)]
print(f"equipo_receiving_yards_season_avg en semana 5 (no debe ver la semana 5 misma): "
      f"{semana_5['equipo_receiving_yards_season_avg'].iloc[0]:.2f}")

suma manual de receiving_yards de CIN, semanas 1-4: 456, entre 4 semanas = 114.00
equipo_receiving_yards_season_avg en semana 5 (no debe ver la semana 5 misma): 114.00


Coinciden: el promedio que ve un receptor de Cincinnati en la semana 5 (114.00) es exactamente el
promedio de las semanas 1 a 4 de su equipo, sin la semana 5 misma.

## 8. Combinaciones, solo con una razón concreta

Cada combinación responde una pregunta que no se puede contestar mirando las variables por separado.

In [14]:
wr = features_exploratorio.agregar_interaccion(
    wr, "target_share", "equipo_receiving_yards_season_avg", nombre="target_share_x_ofensiva_equipo"
)
wr = features_exploratorio.agregar_interaccion(
    wr, "draft_pick", "anios_experiencia", nombre="draft_pick_x_experiencia"
)
wr["cambio_qb_titular_num"] = wr["cambio_qb_titular"].astype("float")
wr = features_exploratorio.agregar_interaccion(
    wr, "cambio_qb_titular_num", "target_share", nombre="cambio_qb_x_target_share"
)

cols_interaccion = ["target_share_x_ofensiva_equipo", "draft_pick_x_experiencia", "cambio_qb_x_target_share"]
wr[cols_interaccion].describe()

,target_share_x_ofensiva_equipo,draft_pick_x_experiencia,cambio_qb_x_target_share
count,22183.000000,23610.000000,22851.000000
mean,20.946759,601.422787,0.018711
std,18.195130,862.296209,0.059859
min,0.000000,0.000000,0.000000
25%,6.184811,48.000000,0.000000
50%,16.888889,252.000000,0.000000
75%,31.521739,765.000000,0.000000
max,131.272727,6149.000000,0.600000


- **`target_share_x_ofensiva_equipo`**: ¿pesa más la misma participación en una ofensiva fuerte que
  en una débil?
- **`draft_pick_x_experiencia`**: ¿el número de draft pesa igual en un novato que en un jugador
  establecido?
- **`cambio_qb_x_target_share`**: ¿un cambio de QB afecta más a los receptores con más
  participación?

Si alguna explica algo que sus dos variables por separado no explican se evalúa en
[`3.2_seleccion_features.ipynb`](3.2_seleccion_features.ipynb).

## Cierre

Funciones construidas y verificadas fila por fila sobre casos reales:

| Función | Módulo | Verificada con |
|---|---|---|
| `calcular_ratios_eficiencia` | `features.py` | Ja'Marr Chase, semana 1 de 2025; vacíos correctos sin pases dirigidos |
| `acotar_variable` | `features_exploratorio.py` | Khalil Shakir, semana 9 de 2025 (`racr` = 43) |
| `agregar_volatilidad_jugador` | `features_exploratorio.py` | Ja'Marr Chase, 2025 |
| `agregar_edad_experiencia` / `agregar_experiencia_no_lineal` | `features.py` | Jugadores de carrera larga (Smith, Fitzgerald) |
| `agregar_draft_pick` | `features.py` | Chase (seleccionado) contra un WR no seleccionado |
| `identificar_qb_titular` / `agregar_cambio_qb_titular` | `data.py` / `features_exploratorio.py` | Cleveland 2024 (cambio de QB) y Ke'Shawn Williams 2025 (traspaso) |
| `agregar_ofensiva_equipo` | `features_exploratorio.py` | Cincinnati 2025, semanas 1-4 contra la 5 |
| `agregar_interaccion` | `features_exploratorio.py` | Tres combinaciones con razón concreta |

`features.py` reúne las funciones que corren cada semana y `features_exploratorio.py`, las que solo
usan los notebooks. Cuáles de todas las variables ayudan a predecir cada resultado se
responde en [`3.2_seleccion_features.ipynb`](3.2_seleccion_features.ipynb).

Anterior: [`2.6_eda_jugador_destacado.ipynb`](2.6_eda_jugador_destacado.ipynb) · Siguiente:
[`3.2_seleccion_features.ipynb`](3.2_seleccion_features.ipynb)